# Phase 08A — 2026 El Niño hydro-shortage resilience

This standalone experiment applies drought shocks to the monthly river-flow proxy table before rebuilding hourly hydro availability with the canonical `build_hydro_availability()` method. It is a caudal-first sensitivity study, not a rainfall forecast or reservoir simulation.

The 2026 chronology uses normal flows in January–August and reduced selected river caudals in September–December. Demand uses the 2025 CENACE SNI annual target of 32,650.66 GWh applied to the existing 2022 hourly shape.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pypsa
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.availability import load_monthly_flows, load_hydro_assignments
from src.phase08a import (apply_monthly_flow_shock, select_caudals, build_2026_demand, rebuild_hydro_profiles, build_scenario_network)
OUT = ROOT / 'results/phase_08a'; FIG = ROOT / 'reports/figures/phase_08a'
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
FLOW_PATH = ROOT / 'data/raw/generation/caudals.csv'
GEN_PATH = ROOT / 'data/processed/generation/generators_2022.csv'
ASSIGN_PATH = ROOT / 'data/interim/generation/hydro_profile_assignments_2022.csv'
REF_PATH = ROOT / 'results/phase_06/network_solved_2022.nc'
for p in (FLOW_PATH, GEN_PATH, ASSIGN_PATH, REF_PATH):
    if not p.exists(): raise FileNotFoundError(p)
print('Phase 08A inputs verified.')

Cannot find header.dxf (GDAL_DATA is not defined)


FileNotFoundError: C:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\results\phase_06d\baseline\network_solved_2022.nc

## Evidence and transformation contract

The model perturbation is applied to `caudals.csv`, not to final `p_max_pu`. Reservoir plants retain the existing three-month rolling-flow rule and quantile normalization. Only the selected caudal series and September–December are shocked. Network topology, installed capacities, non-hydro profiles and interconnection policy remain unchanged.

In [ ]:
flows = load_monthly_flows(FLOW_PATH)
generators = pd.read_csv(GEN_PATH)
assignments = load_hydro_assignments(ASSIGN_PATH)
reference = pypsa.Network(REF_PATH)
selected = select_caudals(generators, assignments, count=4)
selected.to_csv(OUT / 'hydro_scope.csv', index=False)
selected_ids = selected.caudal_id.tolist()
months = (9, 10, 11, 12)
demand = build_2026_demand(reference)
print('Selected caudals:', selected_ids)
print('2026 demand target (GWh):', round(demand.sum().sum()/1000, 2))
display(selected)

In [ ]:
# Audit the caudal shock and rebuilt hourly hydro profiles before any solve.
severities = [0.00, 0.10, 0.20, 0.30, 0.40, 0.50]
profile_cache = {}
caudal_rows = []
for reduction in severities:
    scenario = f'H{int(reduction*100):02d}'
    shocked = apply_monthly_flow_shock(flows, caudal_ids=selected_ids, months=months, reduction=reduction)
    hydro, diagnostics = rebuild_hydro_profiles(generators, shocked, assignments, year=2026)
    profile_cache[scenario] = (shocked, hydro, diagnostics)
    for caudal in selected_ids:
        for month in range(1, 13):
            caudal_rows.append({'scenario':scenario, 'reduction':reduction, 'caudal_id':caudal, 'month':month, 'base_flow':float(flows.loc[caudal, str(month)]), 'scenario_flow':float(shocked.loc[caudal, str(month)]), 'stress_month':month in months})
pd.DataFrame(caudal_rows).to_csv(OUT / 'caudal_scenarios.csv', index=False)
print('Caudal-first hydro profile rebuild complete for', len(severities), 'scenarios.')

## Full-year solves

The following cell runs six 8,760-hour scenarios. It writes a solved NetCDF and solver log for every case. Load shedding is retained as an explicit adequacy diagnostic.

In [ ]:
import gc
replace = True  # Re-solve existing files after the restored 450 MW Colombia cap.
summary_rows = []
for reduction in severities:
    scenario = f'H{int(reduction*100):02d}'
    nc_path = OUT / f'{scenario}.solved.nc'
    if nc_path.exists() and not replace:
        print(f'{scenario}: loading existing result')
        n = pypsa.Network(nc_path)
        status = n.meta.get('solver_status', 'loaded_existing_result')
        termination = n.meta.get('termination_condition', 'loaded_existing_result')
    else:
        _, hydro, diagnostics = profile_cache[scenario]
        n = build_scenario_network(reference, hydro, demand, year=2026)
        log_path = OUT / f'{scenario}_highs.log'
        status, termination = n.optimize(solver_name='highs', solver_options={'threads': 1}, log_fn=str(log_path))
        n.meta.update({'scenario':scenario, 'caudal_reduction':reduction, 'stress_months':list(months), 'solver_status':str(status), 'termination_condition':str(termination)})
        n.export_to_netcdf(nc_path)
    supply = n.generators_t.p.mul(n.generators.sign, axis=1).sum() / 1000
    by_carrier = supply.groupby(n.generators.carrier).sum()
    demand_gwh = float(n.loads_t.p_set.sum().sum()/1000)
    shed = float(by_carrier.get('load_shedding', 0.0))
    row = {'scenario':scenario, 'caudal_reduction':reduction, 'demand_gwh':demand_gwh, 'solver_status':str(status), 'termination_condition':str(termination), 'load_shedding_gwh':shed, 'load_shedding_pct':100*shed/demand_gwh}
    row.update({f'{c}_gwh':float(v) for c,v in by_carrier.items()})
    summary_rows.append(row)
    del n
    gc.collect()
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / 'scenario_summary.csv', index=False)
display(summary[['scenario','caudal_reduction','demand_gwh','solver_status','termination_condition','load_shedding_gwh','load_shedding_pct']])

In [ ]:
# Publication-quality diagnostics.
plt.rcParams.update({'font.family':'Arial','font.sans-serif':['Arial','DejaVu Sans']})
viridis = plt.get_cmap('viridis')
fig, ax = plt.subplots(figsize=(8.5,5)); ax.plot(summary.caudal_reduction*100, summary.load_shedding_gwh, marker='o', color=viridis(.72), linewidth=2); ax.axhline(0, color='black', linewidth=.8); ax.set(xlabel='Selected caudal reduction during Sep–Dec (%)', ylabel='Annual load shedding (GWh)', title='2026 hydro-shortage sensitivity'); ax.grid(alpha=.2); fig.tight_layout(); fig.savefig(FIG/'shedding_vs_caudal_reduction.png', dpi=220); plt.show()
carrier_cols = [c for c in summary.columns if c.endswith('_gwh') and c not in ('demand_gwh','load_shedding_gwh')]
carrier = summary.set_index('scenario')[carrier_cols].rename(columns=lambda c:c.removesuffix('_gwh').replace('_',' ')); carrier.to_csv(OUT/'generation_by_carrier.csv')
fig, ax = plt.subplots(figsize=(10,5)); carrier.plot(ax=ax, color=[viridis(x) for x in np.linspace(.05,.95,len(carrier.columns))]); ax.set(xlabel='Scenario', ylabel='Annual energy (GWh)', title='Energy response to caudal shortage'); ax.legend(bbox_to_anchor=(1.02,1), loc='upper left', frameon=False); ax.grid(alpha=.2); fig.tight_layout(rect=[0,0,.8,1]); fig.savefig(FIG/'generation_response_by_scenario.png', dpi=220); plt.show()

In [ ]:
# Monthly shedding and a representative-week dispatch trace. Reload one result at a time to limit memory use.
monthly_rows=[]
for scenario in [f'H{int(reduction*100):02d}' for reduction in severities]:
    n = pypsa.Network(OUT / f'{scenario}.solved.nc')
    shed = n.generators_t.p.loc[:, n.generators.carrier.eq('load_shedding')].sum(axis=1)
    monthly_rows.extend({'scenario':scenario,'month':int(ts.month),'load_shedding_mwh':float(value)} for ts,value in shed.resample('MS').sum().items())
    del n; gc.collect()
monthly = pd.DataFrame(monthly_rows); monthly.to_csv(OUT/'shedding_by_month.csv', index=False)
fig, ax = plt.subplots(figsize=(10,5)); monthly.pivot(index='month',columns='scenario',values='load_shedding_mwh').plot(ax=ax, marker='o', color=[viridis(x) for x in np.linspace(.05,.95,len(severities))]); ax.set(xlabel='Month in 2026',ylabel='Load shedding (MWh)',title='Monthly shedding under caudal shocks'); ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',frameon=False); ax.grid(alpha=.2); fig.tight_layout(rect=[0,0,.8,1]); fig.savefig(FIG/'shedding_by_month_and_scenario.png',dpi=220); plt.show()
fig, axes = plt.subplots(3,1,figsize=(13,10),sharex=True); style={'hydro':('-','Hydro'),'oil':('--','Oil'),'interconnection_colombia':('-.','Inter COL'),'load_shedding':((0,(3,1,1,1)),'Shedding')}
for ax, scenario in zip(axes, [f'H{int(reduction*100):02d}' for reduction in severities][::2]):
    n = pypsa.Network(OUT / f'{scenario}.solved.nc')
    snaps=n.snapshots[:168]; dispatch=n.generators_t.p.loc[snaps].mul(n.generators.sign,axis=1).groupby(n.generators.carrier,axis=1).sum()
    for carrier,(linestyle,label) in style.items():
        if carrier in dispatch: ax.plot(dispatch.index,dispatch[carrier],linestyle=linestyle,label=label,linewidth=1.5)
    ax.set_title(scenario); ax.set_ylabel('MW'); ax.grid(alpha=.2); ax.legend(bbox_to_anchor=(1.02,1),loc='upper left',frameon=False); del n; gc.collect()
axes[-1].set_xlabel('First 168 snapshots'); fig.tight_layout(rect=[0,0,.82,1]); fig.savefig(FIG/'representative_week_dispatch_by_scenario.png',dpi=220); plt.show()

## Interpretation guardrail

The primary result is the caudal-reduction threshold at which shedding becomes material. A solver status of `optimal` does not mean adequate supply: the load-shedding generators preserve mathematical feasibility. The results are a basin-specific operational sensitivity, not a climate forecast or lifecycle environmental assessment.